<a href="https://colab.research.google.com/github/jiraroj-wir/MUIC-ICCS261-Principles-of-Data-Science/blob/main/Copy_of_loading_legacy_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Loading Legacy Data

*K. Bunchongchit<br>
Last updated on September 24, 2026*

Data in this task contains the daily weather data from the Global Historical Climatology Network for one weather station (MX17004) in Mexico from **1955 to 2011**. The data set has a column for each possible day in the month, and the "element" column indicates if that row contains the minimum or maximum temperature or the amount of percipitation.

Let's load the data file.

***Warning***: Code in this part will be messy as it is showing the actual process of figuring out a solution. Most of the time we show only the "success". So it is not usual to see how chaotic it could be behind the scence. Realizing this will make you more aware when asking the data engineer team to do something because a regular request may not be that simple.

If you are using a desktop IDE, try
<pre>df_wt = pd.read_csv('data/weather.txt', header=0, sep='\t', engine='python')</pre>with the adjusted folder path.

In [1]:
import pandas as pd

weather_URL = 'https://drive.google.com/uc?export=download&id=' + '1d9b_at4SJMlgtzQ04isPjUVzGEm7hYsl'
df_wt = pd.read_csv(weather_URL)
df_wt

,MX000017004195504TMAX 310 I 310 I 310 I 320 I 330 I 320 I 320 I 330 I 330 I 330 I 330 I 320 I 310 I 310 I 320 I 320 I 320 I 310 I 310 I 320 I 320 I 330 I 330 I 330 I 330 I 330 I 330 I 340 I 330 I 320 I-9999
0,MX000017004195504TMIN 150 I 150 I 160 I ...
1,MX000017004195504PRCP 0 I 0 I 0 I ...
2,MX000017004195505TMAX 310 I 310 I 310 I ...
3,MX000017004195505TMIN 200 I 160 I 160 I ...
4,MX000017004195505PRCP 0 I 0 I 0 I ...
...,...
1708,MX000017004201103TMIN-9999 -9999 -9999 -...
1709,MX000017004201103PRCP 0 S-9999 0 S ...
1710,MX000017004201104TMAX-9999 350 S-9999 -...
1711,MX000017004201104TMIN-9999 168 S-9999 -...


There are many problems with this dataset.
- The first data row is turned to be the column labels.
- There is only one column of data.
- Column separators are inconsistent throughout the file, somes are "I", some are "S", and some are just blanks.


So let's check the raw text file to confirm these issues. We will examine some lines at the beginning and the end of this data file.

In [2]:
# https://stackoverflow.com/questions/1393324/given-a-url-to-a-text-file-what-is-the-simplest-way-to-read-the-contents-of-the
import urllib3  # the lib that handles the url stuff

http = urllib3.PoolManager()
response = http.request('GET', weather_URL)
data = response.data.decode('utf-8')

lines = data.split('\n')
print(f'no of lines = {len(lines)}')
print('The first three rows')
print('\n'.join(lines[:3]))
print('The last three rows')
print('\n'.join(lines[-4:]))  # The actual last line is empty

no of lines = 1715
The first three rows
MX000017004195504TMAX  310  I  310  I  310  I  320  I  330  I  320  I  320  I  330  I  330  I  330  I  330  I  320  I  310  I  310  I  320  I  320  I  320  I  310  I  310  I  320  I  320  I  330  I  330  I  330  I  330  I  330  I  330  I  340  I  330  I  320  I-9999   
MX000017004195504TMIN  150  I  150  I  160  I  150  I  160  I  160  I  160  I  160  I  160  I  170  I  170  I  160  I  160  I  160  I  170  I  170  I  160  I  160  I  160  I  160  I  170  I  170  I  170  I  170  I  180  I  190  I  190  I  170  I  180  I  160  I-9999   
MX000017004195504PRCP    0  I    0  I    0  I    0  I    0  I    0  I    0  I    0  I    0  I    0  I    0  I    0  I    0  I    0  I    0  I    0  I    0  I    0  I    0  I    0  I    0  I    0  I    0  I    0  I    0  I    0  I    0  I    0  I    0  I    6  I-9999   
The last three rows
MX000017004201104TMAX-9999     350  S-9999   -9999   -9999   -9999   -9999   -9999   -9999   -9999   -9999   -9999   -9999   -9999

So our observation is correct; the column separators are inconsistent. As the raw file does not contain "\t", our data is not tab-limited either. So we should try the "I" as a separator next. Even though the separators are inconsistent, trying a simple thing normally gives clues what to do next.

However, before we go on and trying to detangle this mess. Let's get a good understand the data in the file.

In [3]:
df_wt = pd.read_csv(weather_URL, header=0, sep=' I ')
df_wt

/var/folders/x1/q5ftwsxd63l56l4dr27_lk2r0000gn/T/ipykernel_46480/790107749.py:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df_wt = pd.read_csv(weather_URL, header=0, sep=' I ')


ParserError: Expected 30 fields in line 4, saw 31. Error could possibly be due to quotes being ignored when a multi-char delimiter is used.

Nope, it fails earlier than expected since Line 5.


What else can I try? As this is a legacy data file, could it be that the typical delimeter convention did not apply? So let's use a string as a delimiter. Of course, this cannot be the only delimiter, but as stated before, let's try simple things first.

In [4]:
df_wt = pd.read_fwf(weather_URL, sep=' I ')
df_wt

,MX000017004195504TMAX,310,I,310.1,I.1,310.2,I.2,320,I.3,330,...,I.24,330.9,I.25,330.10,I.26,340,I 330,I 320,I-9999,Unnamed: 59
0,MX000017004195504TMIN,150,I,150,I,160,I,150,I,160,...,I,190,I,190,I,170,I 180,I 160,I-9999,NaN
1,MX000017004195504PRCP,0,I,0,I,0,I,0,I,0,...,I,0,I,0,I,0,I 0,I 6,I-9999,NaN
2,MX000017004195505TMAX,310,I,310,I,310,I,300,I,300,...,I,310,I,310,I,320,I 310,I 300,I 290,I
3,MX000017004195505TMIN,200,I,160,I,160,I,150,I,150,...,I,180,I,160,I,150,I 170,I 150,I 160,I
4,MX000017004195505PRCP,0,I,0,I,0,I,0,I,0,...,I,0,I,142,I,0,I 54,I 0,I 46,I
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1708,MX000017004201103TMIN,999,NaN,999,NaN,999,NaN,999,NaN,148,...,NaN,162,S,999,NaN,999,-9999,-9999,170,S
1709,MX000017004201103PRCP,0,S,999,NaN,0,S,0,S,0,...,NaN,0,S,999,NaN,999,0,S-9999,0,S
1710,MX000017004201104TMAX,999,NaN,350,S,999,NaN,999,NaN,999,...,NaN,999,NaN,999,NaN,999,-9999,-9999,-9999,NaN
1711,MX000017004201104TMIN,999,NaN,168,S,999,NaN,999,NaN,999,...,NaN,999,NaN,999,NaN,999,-9999,-9999,-9999,NaN


Surprisingly, everything is fit to columns even with line with no "I" as the delimiter. After looking at the raw text again, it could be a fixed-width text file.


Let's try this with a few columns first.

In [5]:
# Warning: Experimental code in progress!
# https://towardsdatascience.com/parsing-fixed-width-text-files-with-pandas-f1db8f737276

colspacing = [(0, 17), (17, 21), (21, 29), (29, None)]
df_temp = pd.read_fwf(weather_URL, colspecs=colspacing, header=None,
            names=['station', 'measurement', 'd1', 'rest'])
df_temp

,station,measurement,d1,rest
0,MX000017004195504,TMAX,310 I,310 I 310 I 320 I 330 I 320 I 320 I...
1,MX000017004195504,TMIN,150 I,150 I 160 I 150 I 160 I 160 I 160 I...
2,MX000017004195504,PRCP,0 I,0 I 0 I 0 I 0 I 0 I 0 I ...
3,MX000017004195505,TMAX,310 I,310 I 310 I 300 I 300 I 300 I 310 I...
4,MX000017004195505,TMIN,200 I,160 I 160 I 150 I 150 I 150 I 160 I...
...,...,...,...,...
1709,MX000017004201103,TMIN,-9999,-9999 -9999 -9999 148 S-9999 -9999 ...
1710,MX000017004201103,PRCP,0 S,-9999 0 S 0 S 0 S-9999 0 ...
1711,MX000017004201104,TMAX,-9999,350 S-9999 -9999 -9999 -9999 -9999 ...
1712,MX000017004201104,TMIN,-9999,168 S-9999 -9999 -9999 -9999 -9999 ...


This is even worse. Now we have the letter 'I' mixed into the values, and the first row of data turn into column labels. OK, let's go for another try without column specifications. Keep fingers crossed while running this!

In [6]:
df_wt = pd.read_fwf(weather_URL, names=['Description']+['c'+ str(i) for i in range(1,64)])
df_wt

,Description,c1,c2,c3,c4,c5,c6,c7,c8,c9,...,c54,c55,c56,c57,c58,c59,c60,c61,c62,c63
0,MX000017004195504TMAX,310,I,310,I,310,I,320,I,330,...,I,340,I 330,I 320,I-9999,NaN,NaN,NaN,NaN,NaN
1,MX000017004195504TMIN,150,I,150,I,160,I,150,I,160,...,I,170,I 180,I 160,I-9999,NaN,NaN,NaN,NaN,NaN
2,MX000017004195504PRCP,0,I,0,I,0,I,0,I,0,...,I,0,I 0,I 6,I-9999,NaN,NaN,NaN,NaN,NaN
3,MX000017004195505TMAX,310,I,310,I,310,I,300,I,300,...,I,320,I 310,I 300,I 290,I,NaN,NaN,NaN,NaN
4,MX000017004195505TMIN,200,I,160,I,160,I,150,I,150,...,I,150,I 170,I 150,I 160,I,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1709,MX000017004201103TMIN,999,NaN,999,NaN,999,NaN,999,NaN,148,...,NaN,999,-9999,-9999,170,S,NaN,NaN,NaN,NaN
1710,MX000017004201103PRCP,0,S,999,NaN,0,S,0,S,0,...,NaN,999,0,S-9999,0,S,NaN,NaN,NaN,NaN
1711,MX000017004201104TMAX,999,NaN,350,S,999,NaN,999,NaN,999,...,NaN,999,-9999,-9999,-9999,NaN,NaN,NaN,NaN,NaN
1712,MX000017004201104TMIN,999,NaN,168,S,999,NaN,999,NaN,999,...,NaN,999,-9999,-9999,-9999,NaN,NaN,NaN,NaN,NaN


Whew... The dataframe now seems to have columns nicely separated. But there are still many junk columns. I manage to find another use of ```pandas.fwf``` from https://sparkbyexamples.com/pandas/pandas-read-text-into-dataframe/. With some more calculations, we generate the column headers and then the array to speicify width of each column.

In [7]:
# https://stackoverflow.com/questions/952914/how-do-i-make-a-flat-list-out-of-a-list-of-lists
import functools
import itertools
import operator

cols = ['station', 'month', 'type'] + functools.reduce(operator.iconcat, [['d'+str(i), 'u' + str(i)] for i in range(1, 32)], [])
print(cols)

['station', 'month', 'type', 'd1', 'u1', 'd2', 'u2', 'd3', 'u3', 'd4', 'u4', 'd5', 'u5', 'd6', 'u6', 'd7', 'u7', 'd8', 'u8', 'd9', 'u9', 'd10', 'u10', 'd11', 'u11', 'd12', 'u12', 'd13', 'u13', 'd14', 'u14', 'd15', 'u15', 'd16', 'u16', 'd17', 'u17', 'd18', 'u18', 'd19', 'u19', 'd20', 'u20', 'd21', 'u21', 'd22', 'u22', 'd23', 'u23', 'd24', 'u24', 'd25', 'u25', 'd26', 'u26', 'd27', 'u27', 'd28', 'u28', 'd29', 'u29', 'd30', 'u30', 'd31', 'u31']


In [8]:
# https://sparkbyexamples.com/pandas/pandas-read-text-into-dataframe/
df_wt = pd.read_fwf(weather_URL,  # the URL from which the data file is being read.
                    header=None,  # This tells pandas that the file does not have a header row
                    widths=[11,6,4] + [7, 1]*31,  # See below
                    names=cols)  # custom column names generated in the previous code cell
df_wt

# This is a list that defines the width of each column in the fixed-width file.
# It specifies the width for the first three columns as 11, 6, and 4 characters
# respectively, and then repeats a pattern of 7 and 1 characters for the next
# 31 pairs of columns. This is crucial for correctly parsing the data into separate columns.

,station,month,type,d1,u1,d2,u2,d3,u3,d4,...,d27,u27,d28,u28,d29,u29,d30,u30,d31,u31
0,MX000017004,195504,TMAX,310,I,310,I,310,I,320,...,330,I,340,I,330,I,320,I,-9999,NaN
1,MX000017004,195504,TMIN,150,I,150,I,160,I,150,...,190,I,170,I,180,I,160,I,-9999,NaN
2,MX000017004,195504,PRCP,0,I,0,I,0,I,0,...,0,I,0,I,0,I,6,I,-9999,NaN
3,MX000017004,195505,TMAX,310,I,310,I,310,I,300,...,310,I,320,I,310,I,300,I,290,I
4,MX000017004,195505,TMIN,200,I,160,I,160,I,150,...,160,I,150,I,170,I,150,I,160,I
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1709,MX000017004,201103,TMIN,-9999,NaN,-9999,NaN,-9999,NaN,-9999,...,-9999,NaN,-9999,NaN,-9999,NaN,-9999,NaN,170,S
1710,MX000017004,201103,PRCP,0,S,-9999,NaN,0,S,0,...,-9999,NaN,-9999,NaN,0,S,-9999,NaN,0,S
1711,MX000017004,201104,TMAX,-9999,NaN,350,S,-9999,NaN,-9999,...,-9999,NaN,-9999,NaN,-9999,NaN,-9999,NaN,-9999,NaN
1712,MX000017004,201104,TMIN,-9999,NaN,168,S,-9999,NaN,-9999,...,-9999,NaN,-9999,NaN,-9999,NaN,-9999,NaN,-9999,NaN


Now it is a successful attmept even though not perfect. Data are in their own columns. There are still two issues to deal with though.
1. There is some mess in the delimeter columns but we are going to discard that anyway.
2. Now the sentinel value from missing data has changed from -9999 to 999. We can change it to NaN. The reason for selecting this value will be discussed on the next notebook.

We will drop the delimeter columns first.

In [9]:
df_wt.drop(['u'+str(i) for i in range(1, 32)], axis=1, inplace=True)
df_wt

,station,month,type,d1,d2,d3,d4,d5,d6,d7,...,d22,d23,d24,d25,d26,d27,d28,d29,d30,d31
0,MX000017004,195504,TMAX,310,310,310,320,330,320,320,...,330,330,330,330,330,330,340,330,320,-9999
1,MX000017004,195504,TMIN,150,150,160,150,160,160,160,...,170,170,170,180,190,190,170,180,160,-9999
2,MX000017004,195504,PRCP,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,6,-9999
3,MX000017004,195505,TMAX,310,310,310,300,300,300,310,...,330,340,350,330,310,310,320,310,300,290
4,MX000017004,195505,TMIN,200,160,160,150,150,150,160,...,170,190,190,190,180,160,150,170,150,160
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1709,MX000017004,201103,TMIN,-9999,-9999,-9999,-9999,148,-9999,-9999,...,-9999,-9999,-9999,-9999,162,-9999,-9999,-9999,-9999,170
1710,MX000017004,201103,PRCP,0,-9999,0,0,0,-9999,0,...,0,-9999,0,-9999,0,-9999,-9999,0,-9999,0
1711,MX000017004,201104,TMAX,-9999,350,-9999,-9999,-9999,-9999,-9999,...,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999
1712,MX000017004,201104,TMIN,-9999,168,-9999,-9999,-9999,-9999,-9999,...,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999


In [10]:
df_wt.info()

<class 'pandas.DataFrame'>
RangeIndex: 1714 entries, 0 to 1713
Data columns (total 34 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   station  1714 non-null   str  
 1   month    1714 non-null   int64
 2   type     1714 non-null   str  
 3   d1       1714 non-null   str  
 4   d2       1714 non-null   str  
 5   d3       1714 non-null   str  
 6   d4       1714 non-null   str  
 7   d5       1714 non-null   str  
 8   d6       1714 non-null   str  
 9   d7       1714 non-null   str  
 10  d8       1714 non-null   str  
 11  d9       1714 non-null   str  
 12  d10      1714 non-null   str  
 13  d11      1714 non-null   str  
 14  d12      1714 non-null   int64
 15  d13      1714 non-null   int64
 16  d14      1714 non-null   int64
 17  d15      1714 non-null   int64
 18  d16      1714 non-null   int64
 19  d17      1714 non-null   str  
 20  d18      1714 non-null   str  
 21  d19      1714 non-null   str  
 22  d20      1714 non-null   str  
 23 

In [11]:
for col_name in df_wt.columns[3:]:
    # Convert to numeric first, coercing any non-numeric values to NaN
    df_wt[col_name] = pd.to_numeric(df_wt[col_name], errors='coerce')
    # Then convert to nullable integer type to handle NaN values
    df_wt[col_name] = df_wt[col_name].astype('Int64')

df_wt

,station,month,type,d1,d2,d3,d4,d5,d6,d7,...,d22,d23,d24,d25,d26,d27,d28,d29,d30,d31
0,MX000017004,195504,TMAX,310,310,310,320,330,320,320,...,330,330,330,330,330,330,340,330,320,-9999
1,MX000017004,195504,TMIN,150,150,160,150,160,160,160,...,170,170,170,180,190,190,170,180,160,-9999
2,MX000017004,195504,PRCP,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,6,-9999
3,MX000017004,195505,TMAX,310,310,310,300,300,300,310,...,330,340,350,330,310,310,320,310,300,290
4,MX000017004,195505,TMIN,200,160,160,150,150,150,160,...,170,190,190,190,180,160,150,170,150,160
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1709,MX000017004,201103,TMIN,-9999,-9999,-9999,-9999,148,-9999,-9999,...,-9999,-9999,-9999,-9999,162,-9999,-9999,-9999,-9999,170
1710,MX000017004,201103,PRCP,0,-9999,0,0,0,-9999,0,...,0,-9999,0,-9999,0,-9999,-9999,0,-9999,0
1711,MX000017004,201104,TMAX,-9999,350,-9999,-9999,-9999,-9999,-9999,...,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999
1712,MX000017004,201104,TMIN,-9999,168,-9999,-9999,-9999,-9999,-9999,...,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999


In [12]:
df_wt.info()

<class 'pandas.DataFrame'>
RangeIndex: 1714 entries, 0 to 1713
Data columns (total 34 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   station  1714 non-null   str  
 1   month    1714 non-null   int64
 2   type     1714 non-null   str  
 3   d1       1710 non-null   Int64
 4   d2       1713 non-null   Int64
 5   d3       1713 non-null   Int64
 6   d4       1711 non-null   Int64
 7   d5       1713 non-null   Int64
 8   d6       1713 non-null   Int64
 9   d7       1712 non-null   Int64
 10  d8       1712 non-null   Int64
 11  d9       1713 non-null   Int64
 12  d10      1711 non-null   Int64
 13  d11      1713 non-null   Int64
 14  d12      1714 non-null   Int64
 15  d13      1714 non-null   Int64
 16  d14      1714 non-null   Int64
 17  d15      1714 non-null   Int64
 18  d16      1714 non-null   Int64
 19  d17      1713 non-null   Int64
 20  d18      1712 non-null   Int64
 21  d19      1713 non-null   Int64
 22  d20      1712 non-null   Int64
 23 

Lastly, we fix the sentinel value to be the popular choice for DataFrame.

In [13]:
import numpy as np

df_wt.replace(-9999, np.nan, inplace=True)
df_wt

,station,month,type,d1,d2,d3,d4,d5,d6,d7,...,d22,d23,d24,d25,d26,d27,d28,d29,d30,d31
0,MX000017004,195504,TMAX,310,310,310,320,330,320,320,...,330,330,330,330,330,330,340,330,320,<NA>
1,MX000017004,195504,TMIN,150,150,160,150,160,160,160,...,170,170,170,180,190,190,170,180,160,<NA>
2,MX000017004,195504,PRCP,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,6,<NA>
3,MX000017004,195505,TMAX,310,310,310,300,300,300,310,...,330,340,350,330,310,310,320,310,300,290
4,MX000017004,195505,TMIN,200,160,160,150,150,150,160,...,170,190,190,190,180,160,150,170,150,160
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1709,MX000017004,201103,TMIN,<NA>,<NA>,<NA>,<NA>,148,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,162,<NA>,<NA>,<NA>,<NA>,170
1710,MX000017004,201103,PRCP,0,<NA>,0,0,0,<NA>,0,...,0,<NA>,0,<NA>,0,<NA>,<NA>,0,<NA>,0
1711,MX000017004,201104,TMAX,<NA>,350,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1712,MX000017004,201104,TMIN,<NA>,168,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
